In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_excel("flight_components_condition_augmented_10000.xlsx")

df.head()

In [ ]:
print(df.shape)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.duplicated().sum())

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    y='Condition Status',
    order=df['Condition Status'].value_counts().index
)

plt.title("Condition Status Distribution")
plt.xlabel("Count")
plt.ylabel("Condition Status")

plt.show()

In [ ]:
num_cols = [
    'Flight Hours',
    'Cycles',
    'Wear Level (%)',
    'Remaining Life (%)'
]

df[num_cols].hist(figsize=(10,8))

plt.show()

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()

In [ ]:
numeric_df = df[num_cols]

plt.figure(figsize=(6,5))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.show()

In [ ]:
df = df.drop(
    columns=[
        'Component ID',
        'Serial Number'
    ]
)

In [ ]:
date_cols = [
    'Install Date',
    'Last Inspection Date',
    'Next Inspection Due'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

    df[col] = df[col].map(pd.Timestamp.toordinal)

In [ ]:
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

In [ ]:
X = df.drop("Condition Status", axis=1)

y = df["Condition Status"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy :",accuracy)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
importance = pd.DataFrame({

    'Feature': X.columns,
    'Importance': rf.feature_importances_

})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

In [ ]:
plt.figure(figsize=(8,6))

sns.barplot(
    data=importance,
    x='Importance',
    y='Feature'
)

plt.title("Feature Importance")
plt.show()